# My Private Knowledge Worker

## Week 5 Challenge: A personal AI assistant that knows my documents

### Architecture:
1. **Ingest**: Load all markdown files from `knowledge-base/` folders
2. **Chunk**: Use an LLM to semantically split documents into overlapping chunks (Advanced RAG)
3. **Embed**: Vectorize chunks using OpenAI's `text-embedding-3-small` and store in Chroma
4. **Retrieve + Rerank**: Fetch relevant chunks, then use LLM to rerank by relevance
5. **Answer**: Query rewriting → retrieval → reranking → augmented generation
6. **UI**: Gradio ChatInterface with context display

### Setup:
- Create a `.env` file with `OPENAI_API_KEY=sk-...`
- Optionally add `OPENROUTER_API_KEY=sk-or-...` for multi-model support
- Put your own `.md` files in `knowledge-base/` subfolders
- Run all cells!

In [ ]:
# Cell 1: Imports and Setup

import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
import gradio as gr

load_dotenv(override=True)

# --- Configuration ---
MODEL = "gpt-4.1-nano"                        # Cheap model for chunking, reranking, query rewriting
ANSWER_MODEL = "gpt-4.1-nano"                  # Model for final answer generation
EMBEDDING_MODEL = "text-embedding-3-small"     # OpenAI embedding model (cheap + good)
DB_NAME = "my_vector_db"                       # Chroma database directory
COLLECTION_NAME = "my_docs"                    # Chroma collection name
KNOWLEDGE_BASE_PATH = Path("knowledge-base")   # Folder containing your documents
AVERAGE_CHUNK_SIZE = 500                       # Target characters per chunk
RETRIEVAL_K = 10                               # How many chunks to retrieve

# --- OpenAI Client ---
openai_client = OpenAI()

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please add it to your .env file")

## Step 1: Pydantic Models for Structured Output

We define exactly what a "chunk" looks like so the LLM returns structured JSON.

In [ ]:
# Cell 2: Pydantic Models

class Chunk(BaseModel):
    """A single chunk produced by the LLM from a document."""
    headline: str = Field(description="A brief heading for this chunk that captures the key topic")
    summary: str = Field(description="A few sentences summarizing the chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk, exactly as written")


class Chunks(BaseModel):
    """A list of chunks that together represent an entire document."""
    chunks: list[Chunk]


class RankOrder(BaseModel):
    """The reranked order of chunk IDs from most to least relevant."""
    order: list[int] = Field(
        description="Chunk IDs ordered from most relevant to least relevant"
    )


class Result(BaseModel):
    """A retrieved chunk with its content and metadata (like LangChain's Document)."""
    page_content: str
    metadata: dict

print("Pydantic models defined.")

## Step 2: Load Documents from the Knowledge Base

This is our homemade version of LangChain's DirectoryLoader.

In [ ]:
# Cell 3: Document Loading

def fetch_documents():
    """Load all .md files from knowledge-base subfolders."""
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        if not folder.is_dir():
            continue
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({
                    "type": doc_type,
                    "source": file.as_posix(),
                    "text": f.read()
                })
    print(f"Loaded {len(documents)} documents")
    return documents

documents = fetch_documents()

# Quick preview
for doc in documents:
    print(f"  [{doc['type']}] {doc['source']} ({len(doc['text']):,} chars)")

## Step 3: LLM-Powered Semantic Chunking (Advanced RAG)

Instead of fixed-size splitting (RecursiveCharacterTextSplitter), we ask the LLM
to intelligently divide each document into overlapping chunks with headlines and summaries.

In [ ]:
# Cell 4: Semantic Chunking with LLM

def make_chunking_prompt(document):
    """Create a prompt that asks the LLM to split a document into chunks."""
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and split it into overlapping chunks for a KnowledgeBase.

The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions.
Divide the document so the entire content is covered - don't leave anything out.
This document should probably be split into {how_many} chunks, but use more or less as appropriate.
There should be about 25% overlap between chunks (roughly 50 words) for best retrieval.

For each chunk, provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""


def process_document(document):
    """Send one document to the LLM and get back structured chunks."""
    messages = [{"role": "user", "content": make_chunking_prompt(document)}]
    response = openai_client.chat.completions.parse(
        model=MODEL,
        messages=messages,
        response_format=Chunks        # Structured output: forces valid JSON matching our Pydantic model
    )
    reply = response.choices[0].message.content
    doc_chunks = Chunks.model_validate_json(reply).chunks
    
    # Convert each Chunk into a Result (with headline + summary prepended to content)
    results = []
    for chunk in doc_chunks:
        content = chunk.headline + "\n\n" + chunk.summary + "\n\n" + chunk.original_text
        metadata = {"source": document["source"], "type": document["type"]}
        results.append(Result(page_content=content, metadata=metadata))
    return results


def create_chunks(documents):
    """Process all documents into chunks."""
    all_chunks = []
    for doc in tqdm(documents, desc="Chunking documents"):
        all_chunks.extend(process_document(doc))
    return all_chunks


chunks = create_chunks(documents)
print(f"\nCreated {len(chunks)} total chunks")
print(f"\nExample chunk:\n{chunks[0].page_content[:300]}...")

## Step 4: Create Embeddings and Store in Chroma

We embed all chunks using OpenAI's embedding model and store them in ChromaDB.

In [ ]:
# Cell 5: Vectorize and Store

def create_vectorstore(chunks):
    """Embed all chunks and store in Chroma."""
    chroma = PersistentClient(path=DB_NAME)
    
    # Delete existing collection if it exists (fresh start)
    if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(COLLECTION_NAME)
    
    # Get embeddings from OpenAI
    texts = [chunk.page_content for chunk in chunks]
    print(f"Embedding {len(texts)} chunks...")
    embeddings_response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    vectors = [e.embedding for e in embeddings_response.data]
    
    # Store in Chroma
    collection = chroma.get_or_create_collection(COLLECTION_NAME)
    ids = [str(i) for i in range(len(chunks))]
    metadatas = [chunk.metadata for chunk in chunks]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metadatas)
    
    print(f"Vectorstore created with {collection.count()} documents")
    return chroma, collection


chroma, collection = create_vectorstore(chunks)

## Step 5: Retrieval + Reranking

Two advanced RAG techniques:
1. **Retrieval**: Embed the query, find the K nearest chunks in Chroma
2. **Reranking**: Ask an LLM to reorder those chunks by actual relevance to the question

In [ ]:
# Cell 6: Retrieval and Reranking

def fetch_context_unranked(question):
    """Embed the question and retrieve the top-K closest chunks from Chroma."""
    query_embedding = openai_client.embeddings.create(
        model=EMBEDDING_MODEL, input=[question]
    ).data[0].embedding
    
    results = collection.query(query_embeddings=[query_embedding], n_results=RETRIEVAL_K)
    
    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append(Result(page_content=doc, metadata=meta))
    return retrieved


def rerank(question, chunks):
    """Use an LLM to reorder retrieved chunks by relevance to the question."""
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of chunks retrieved from a knowledge base.
Rank the chunks by relevance to the question, most relevant first.
Reply only with the ranked list of chunk IDs. Include ALL chunk IDs.
"""
    user_prompt = f"Question: {question}\n\nRank these chunks by relevance:\n\n"
    for i, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {i + 1}:\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the ranked chunk IDs."
    
    response = openai_client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=RankOrder     # Structured output
    )
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]


def fetch_context(question):
    """Retrieve and rerank chunks for a question."""
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)


# Quick test
test_results = fetch_context("What is RAG?")
print(f"Retrieved and reranked {len(test_results)} chunks")
print(f"Top result preview: {test_results[0].page_content[:150]}...")

## Step 6: Query Rewriting + Answer Generation

Before retrieving, we rewrite the user's question to be more specific.
Then we build the final RAG prompt and generate the answer.

In [ ]:
# Cell 7: Query Rewriting and Answer Generation

SYSTEM_PROMPT = """
You are a helpful, knowledgeable assistant.
You answer questions using the provided context from the user's personal knowledge base.
Your answers are evaluated for accuracy, relevance, and completeness.
If you don't know the answer, say so honestly.

Context from the knowledge base:
{context}

Answer the user's question accurately and completely based on this context.
"""


def rewrite_query(question, history=[]):
    """Rewrite the user's question to be more specific for retrieval."""
    message = f"""
You are about to search a personal knowledge base to answer the user's question.

Conversation history so far:
{history}

User's current question:
{question}

Respond ONLY with a single, refined search query - short and specific.
Focus on the key details that will surface relevant content.
IMPORTANT: Respond ONLY with the query, nothing else.
"""
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": message}]
    )
    return response.choices[0].message.content


def answer_question(question, history=[]):
    """Full RAG pipeline: rewrite → retrieve → rerank → generate."""
    # Step 1: Rewrite the query for better retrieval
    query = rewrite_query(question, history)
    print(f"Rewritten query: {query}")
    
    # Step 2: Retrieve and rerank
    context_chunks = fetch_context(query)
    
    # Step 3: Build the augmented prompt
    context = "\n\n".join(
        f"[Source: {c.metadata['source']}]\n{c.page_content}" for c in context_chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]
    
    # Step 4: Generate the answer
    response = openai_client.chat.completions.create(model=ANSWER_MODEL, messages=messages)
    return response.choices[0].message.content, context_chunks


# Quick test
answer, ctx = answer_question("What tools did I use in my projects?")
print(f"\nAnswer:\n{answer}")

## Step 7: Gradio UI

A polished chat interface with a context panel showing what chunks were retrieved.

In [ ]:
# Cell 8: Gradio Chat UI

def format_context(context_chunks):
    """Format retrieved chunks as HTML for the context panel."""
    result = "<h3 style='color: #2563eb;'>Retrieved Context</h3>\n\n"
    for i, doc in enumerate(context_chunks[:5]):  # Show top 5
        result += f"<b style='color: #2563eb;'>Chunk {i+1} - Source: {doc.metadata['source']}</b>\n\n"
        result += doc.page_content[:300] + "...\n\n---\n\n"
    return result


def chat(history):
    """Process the latest message in the chat history."""
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)


def add_message(message, history):
    """Add user message to chat history."""
    return "", history + [{"role": "user", "content": message}]


# Build the UI
theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="My Knowledge Worker", theme=theme) as ui:
    gr.Markdown("# 🧠 My Knowledge Worker\nAsk me anything about your documents!")
    
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="Chat", height=500, type="messages", show_copy_button=True)
            msg = gr.Textbox(label="Your Question", placeholder="Ask anything about your knowledge base...", show_label=False)
        
        with gr.Column(scale=1):
            context_display = gr.Markdown(
                value="*Retrieved context will appear here after you ask a question.*",
                label="Retrieved Context"
            )
    
    msg.submit(add_message, inputs=[msg, chatbot], outputs=[msg, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, context_display]
    )

ui.launch(inbrowser=True)